# classic_control_analysis

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load the stored DQN runs for each environment, skipping any not stored yet.
ENVIRONMENTS = ["mountaincar", "acrobot", "cartpole"]

results = {
    environment: load_results(EXPERIMENT, f"dqn_{environment}")
    for environment in ENVIRONMENTS
}
results = {
    environment: runs for environment, runs in results.items() if runs is not None
}

In [ ]:
# Compute each environment's mean return over steps with a 95% bootstrap CI.
curves = {}
for environment, runs in results.items():
    steps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[environment] = (steps, mean_ci(returns))

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (24, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
DQN_COLOR = "tab:blue"
# Only the left panel carries the legend, since the panels share colors.
LEGEND = {
    "loc": "center right",
    "bbox_to_anchor": (1.0, 0.35),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
YLABEL = "Return"
TITLES = {"mountaincar": "MountainCar", "acrobot": "Acrobot", "cartpole": "Cartpole"}
YLIMS = {"mountaincar": (-1000, 0), "acrobot": (-500, 0), "cartpole": (0, 500)}

In [ ]:
# Plot every environment's panel side by side. Each panel has its own y-axis, so
# each is labelled.
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, len(ENVIRONMENTS))
    for ax, environment in zip(axes, ENVIRONMENTS, strict=True):
        ax.set_box_aspect(BOX_ASPECT)
        ax.set_title(TITLES[environment])
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
        ax.set_ylim(YLIMS[environment])
        ax.set_yticks(YLIMS[environment])
        if environment not in curves:
            continue
        steps, (mean, low, high) = curves[environment]
        ax.fill_between(steps, low, high, color=DQN_COLOR, alpha=BAND_ALPHA)
        ax.plot(steps, mean, color=DQN_COLOR)
        ax.set_xticks([0, steps[-1]], ["0", f"{steps[-1] // 1000}K"])
    fig.tight_layout()
    handle = Line2D([], [], color=DQN_COLOR, **LEGEND_MARKER)
    axes[0].legend([handle], ["DQN"], **LEGEND)
figures = {"classic_control_dqn_curves": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")